In [1]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
!pip install evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


In [6]:
from datasets import load_dataset
from transformers import BlipProcessor, BlipForConditionalGeneration, TrainingArguments, Trainer
import torch
import os

# ==========================================
# Step 1: 加载全量数据 [cite: 32, 33]
# ==========================================
# 加载 fashion 数据集。该数据集包含约 4.4 万个样本 [cite: 28, 33]。
raw_dataset = load_dataset("ashraq/fashion-product-images-small", split='train')

# 进行 90/10 的训练/测试切分 [cite: 31]
dataset = raw_dataset.train_test_split(test_size=0.1)
train_ds = dataset['train']
test_ds = dataset['test']

# ==========================================
# Step 2: 图像与文本预处理 (针对 BLIP) [cite: 8]
# ==========================================
# 使用 Hugging Face 的 BLIP 专用处理器 [cite: 35]
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")

def transform(example_batch):
    # 将图像转换为 RGB 模式
    images = [x.convert("RGB") for x in example_batch['image']]
    # 将 productDisplayName（如 "Men Navy Blue Shirt"）作为微调的目标描述
    captions = [x for x in example_batch['productDisplayName']]

    # 图像与文本联合编码。return_tensors="pt" 确保输出为 PyTorch 张量。
    inputs = processor(images=images, text=captions, return_tensors="pt", padding="max_length", truncation=True)

    # 💡 核心逻辑：BLIP 模型需要 'labels' 字段来计算训练 Loss。
    # 在生成任务中，labels 通常就是输入的 input_ids。
    inputs["labels"] = inputs["input_ids"]
    return inputs

# 使用 set_transform 进行实时转换，节省内存压力
train_ds.set_transform(transform)
test_ds.set_transform(transform)

# ==========================================
# Step 3: 定义模型与训练参数 [cite: 34, 39]
# ==========================================
# 加载预训练的 BLIP 模型 [cite: 35]
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

training_args = TrainingArguments(
    output_dir="./blip-fashion-finetuned",
    remove_unused_columns=False,

    # 💡 核心修改 1: 降低 Batch Size
    # 从 8 降到 4 甚至 2。如果还是报错，请改为 2。
    per_device_train_batch_size=4,

    # 💡 核心修改 2: 梯度累积 (Gradient Accumulation)
    # 这可以在显存受限时模拟大 Batch 的效果。
    # 比如 batch_size=4 且 steps=2，等效于原来的 batch_size=8。
    gradient_accumulation_steps=2,

    # 💡 核心修改 3: 强制释放缓存
    # 在 TrainingArguments 中添加这个参数（需要 transformers 较新版本）
    # 或者在训练前运行 torch.cuda.empty_cache()

    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    num_train_epochs=3,
    learning_rate=5e-5,
    fp16=True, # 继续保持半精度以节省显存
    push_to_hub=True,
    load_best_model_at_end=True,
    report_to="none"
)

# ==========================================
# Step 4: 执行训练
# ==========================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
)

# 🚀 开始训练。务必保留 Notebook 的输出结果，这是评分的一部分。
trainer.train()

# ==========================================
# Step 5: 显式保存模型
# ==========================================
# 保存模型和预处理器，以便后续在 Streamlit App 中加载 [cite: 23, 71]。
trainer.save_model("./blip-fashion-finetuned")
processor.save_pretrained("./blip-fashion-finetuned")

print("✅ Pipeline 1: BLIP 模型微调完成！专业描述生成能力已就绪。")

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 13.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.13 GiB is allocated by PyTorch, and 287.53 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
from datasets import load_dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TrainingArguments, Trainer, DataCollatorForLanguageModeling
import torch
import numpy as np
import evaluate
import os

# ==========================================
# Step 1: 加载全量数据 [cite: 13, 32]
# ==========================================
# 该数据集专门用于将产品名称转化为广告语，非常符合业务场景
raw_dataset = load_dataset("llm-wizard/Product-Descriptions-and-Ads", split='train')

# 💡 逻辑：采用 90/10 的比例切分，确保有足够的验证数据评估“准确性”
split_ds = raw_dataset.train_test_split(test_size=0.1)

# ==========================================
# Step 2: 准备分词器 (Tokenizer)
# ==========================================
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
# 💡 必须设置填充字符，GPT-2 默认没有 pad_token
tokenizer.pad_token = tokenizer.eos_token

# ==========================================
# Step 3: 预处理函数 (Prompt Engineering)
# ==========================================
def tokenize_function(example):
    # 从数据集中提取字段。注意：需确保字段名与数据集完全匹配
    product = example.get('Product Name', '')
    ad = example.get('Ad', '')

    # 构造 Prompt：引导模型学习从 "Product" 到 "Ad" 的映射逻辑 [cite: 10]
    full_text = f"Product: {product} \n Ad: {ad} {tokenizer.eos_token}"

    return tokenizer(full_text, truncation=True, max_length=128)

# 移除原始列，仅保留张量化的 input_ids
tokenized_train = split_ds['train'].map(tokenize_function, remove_columns=raw_dataset.column_names)
tokenized_eval = split_ds['test'].map(tokenize_function, remove_columns=raw_dataset.column_names)

# ==========================================
# Step 4: 定义准确率指标 (Token-level Accuracy)
# ==========================================
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # 💡 关键逻辑：在 Causal LM (GPT-2) 中，预测值是 logits 的最后一个维度
    # 且预测结果通常相对于标签向右偏移一位
    predictions = np.argmax(logits, axis=-1)

    # 仅计算非 Padding 部分的准确率，-100 是 PyTorch 默认的忽略索引
    mask = labels != -100
    labels_eval = labels[mask]
    preds_eval = predictions[mask]

    return accuracy_metric.compute(predictions=preds_eval, references=labels_eval)

# ==========================================
# Step 5: 训练参数设置 (资源优化策略)
# ==========================================
training_args = TrainingArguments(
    output_dir="./gpt2-ad-finetuned", #

    # 💡 内存优化：针对 T4 GPU 预防 OOM (CUDA Out of Memory)
    per_device_train_batch_size=4,   # 减小 Batch Size
    gradient_accumulation_steps=2,   # 梯度累积，模拟更大的 Batch
    fp16=True,                      # 半精度训练，节省显存

    num_train_epochs=5,             # 增加轮数以确保文案的流畅度
    learning_rate=5e-5,
    weight_decay=0.01,

    # 💡 评估与保存对齐：修复 ValueError
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,                  # 必须是 eval_steps 的倍数

    load_best_model_at_end=True,    # 自动保存最优模型 [cite: 89]
    metric_for_best_model="accuracy",
    save_total_limit=2,
    push_to_hub=True,               # 上传至 Hugging Face 以供 App 调用 [cite: 22]
    report_to="none"
)

# ==========================================
# Step 6: 执行训练
# ==========================================
model = GPT2LMHeadModel.from_pretrained("gpt2")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    # DataCollator 会自动处理 Padding 并生成 labels
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    compute_metrics=compute_metrics,
)

# 开始训练。保留 Notebook 输出以备查验
trainer.train()

# ==========================================
# Step 7: 显式保存
# ==========================================
trainer.save_model("./gpt2-ad-finetuned")
tokenizer.save_pretrained("./gpt2-ad-finetuned")

print("✅ Pipeline 2: GPT-2 模型训练完成！营销文案生成能力已就绪。")

README.md:   0%|          | 0.00/897 [00:00<?, ?B/s]

data/train-00000-of-00001-36bd1b172faedc(…):   0%|          | 0.00/19.3k [00:00<?, ?B/s]

data/test-00000-of-00001-da7439d21cc2d71(…):   0%|          | 0.00/5.61k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10 [00:00<?, ? examples/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...netuned/training_args.bin: 100%|##########| 5.14kB / 5.14kB            

  ...netuned/model.safetensors:   8%|8         | 40.0MB /  498MB            

✅ Pipeline 2: GPT-2 模型训练完成并已保存至 ./gpt2-ad-finetuned


In [ ]:
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, TrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import torch
import numpy as np
import evaluate

# ==========================================
# Step 1: 加载全量数据 [cite: 32]
# ==========================================
raw_dataset = load_dataset("llm-wizard/Product-Descriptions-and-Ads", split='train')
split_ds = raw_dataset.train_test_split(test_size=0.1)

# ==========================================
# Step 2: 准备分词器 (T5 专用)
# ==========================================
tokenizer = T5Tokenizer.from_pretrained("t5-base")

# ==========================================
# Step 3: 预处理函数 (T5 需要特定的 Prefix)
# ==========================================
def preprocess_function(examples):
    # 💡 T5 逻辑：给输入加上指令前缀 (Task Prefix)
    inputs = ["generate marketing ad: " + (doc if doc else "") for doc in examples["Product Name"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")

    # 💡 设置目标文本 (Labels)
    labels = tokenizer(text_target=examples["Ad"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = split_ds['train'].map(preprocess_function, batched=True, remove_columns=raw_dataset.column_names)
tokenized_eval = split_ds['test'].map(preprocess_function, batched=True, remove_columns=raw_dataset.column_names)

# ==========================================
# Step 4: 定义准确率指标
# ==========================================
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # 对预测结果取最高概率的 Token
    predictions = np.argmax(predictions[0] if isinstance(predictions, tuple) else predictions, axis=-1)

    # 过滤掉 Padding 符号 (-100)
    mask = labels != -100
    return accuracy_metric.compute(predictions=predictions[mask], references=labels[mask])

# ==========================================
# Step 5: 训练参数设置 (Seq2Seq 专用)
# ==========================================
training_args = TrainingArguments(
    output_dir="./t5-ad-finetuned",
    predict_with_generate=True,     # T5 评估时需要开启生成模式
    per_device_train_batch_size=4,   # T5-base 较重，建议设为 4 预防 OOM
    gradient_accumulation_steps=2,
    fp16=True,                      # 资源优化
    num_train_epochs=5,
    learning_rate=5e-4,             # T5 通常需要稍大一点的学习率
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=True,
    report_to="none"
)

# ==========================================
# Step 6: 初始化模型与 Trainer
# ==========================================
model = T5ForConditionalGeneration.from_pretrained("t5-base")

# T5 使用 Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
    compute_metrics=compute_metrics,
)

# 开始训练
trainer.train()

# ==========================================
# Step 7: 显式保存
# ==========================================
trainer.save_model("./t5-ad-finetuned")
tokenizer.save_pretrained("./t5-ad-finetuned")
print("✅ Pipeline 2: T5 模型训练完成！")

In [ ]:
import os
from google.colab import files

# ==========================================
# Step 1: 定义文件夹结构 (按课程要求) [cite: 43, 61]
# ==========================================
# 请修改为你自己的组号和学号
group_id = "Group01"
leader_id = "07123456"
root_folder = f"{group_id}_{leader_id}"

# 创建项目要求的子目录 [cite: 61-81]
paths = [
    f"{root_folder}/{group_id}_documentation",
    f"{root_folder}/{root_folder}_program/Python_notebooks",
    f"{root_folder}/{root_folder}_program/GitHub_App_Files",
    f"{root_folder}/{group_id}_Dataset_files/Fine-tuned_Model_files",
    f"{root_folder}/{group_id}_presentation"
]

for path in paths:
    os.makedirs(path, exist_ok=True)

# ==========================================
# Step 2: 保存三个微调模型 (仿照截图方法)
# ==========================================
model_base_path = f"{root_folder}/{group_id}_Dataset_files/Fine-tuned_Model_files"

# 1. 保存 BLIP 模型
trainer_blip.save_model(f"{model_base_path}/blip-fashion-finetuned")
processor_blip.save_pretrained(f"{model_base_path}/blip-fashion-finetuned")

# 2. 保存 GPT-2 模型
trainer_gpt2.save_model(f"{model_base_path}/gpt2-ad-finetuned")
tokenizer_gpt2.save_pretrained(f"{model_base_path}/gpt2-ad-finetuned")

# 3. 保存 T5 模型
trainer_t5.save_model(f"{model_base_path}/t5-ad-finetuned")
tokenizer_t5.save_pretrained(f"{model_base_path}/t5-ad-finetuned")

print("✅ 所有模型已保存至指定目录结构中。")

# ==========================================
# Step 3: 打包并下载 (完全遵照截图语法)
# ==========================================
# 使用命令行压缩整个项目文件夹 [cite: 43]
!zip -r /content/{root_folder}.zip /content/{root_folder}

# 弹出下载窗口
files.download(f"/content/{root_folder}.zip")